# 1. Khởi tạo môi trường & Đọc dữ liệu

In [4]:

# export
import re
import pandas as pd
from helpers.get_configs import get_config

RAW_DATA_PATH = get_config('dirs.yaml', ['raw_data', 'score_blocks'], as_path=True)
CLEANED_DATA_PATH = get_config('dirs.yaml', ['cleaned_data', 'score_blocks'], as_path=True)
SCRIPT_OUTPUT_PATH = get_config('dirs.yaml', ['unit_scripts', 'score_blocks'], as_path=True)

# Đọc file CSV vào DataFrame
df = pd.read_parquet(RAW_DATA_PATH)


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\UNIDOCS\\IS405.R11-Big-Data\\data-raw\\admission-cutoff-score\\score-blocks.csv'

In [ ]:
# Khảo sát thông tin tổng quan của dữ liệu
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   code      280 non-null    str  
 1   subject1  280 non-null    str  
 2   subject2  280 non-null    str  
 3   subject3  280 non-null    str  
 4   subject4  1 non-null      str  
dtypes: str(5)
memory usage: 21.8 KB


,code,subject1,subject2,subject3,subject4
0,D01,Toán,Ngữ văn,Tiếng anh,NaN
1,A00,Toán,Vật lí,Hóa học,NaN
2,A01,Toán,Vật lí,Tiếng anh,NaN
3,D07,Toán,Hóa học,Tiếng anh,NaN
4,B00,Toán,Hóa học,Sinh học,NaN


# 2. Kiểm tra giá trị rỗng

In [ ]:
# Kiểm tra số lượng giá trị rỗng trên các cột
df.isna().sum()

code          0
subject1      0
subject2      0
subject3      0
subject4    279
dtype: int64

**Kết luận**: Các cột bị thiếu dữ liệu đều thuộc về `subject4` (một tổ hợp môn có thể có 4 môn học hoặc 3 môn học), dữ liệu hợp lý, nên không cần bổ sung / lược bỏ các hàng này.

# 3. Chuẩn hóa dữ liệu

In [ ]:
# export
import unicodedata

SUBJECT_COLS = ['subject1', 'subject2', 'subject3', 'subject4']

# Ánh xạ áp dụng sau khi đã chuẩn hóa (chữ thường, dấu thanh kiểu mới, lý -> lí)
SUBJECT_MAP = {
    'gdktpl': 'gdkt&pl',
    'hóa': 'hóa học',
    'lí': 'vật lí',
    'địa': 'địa lí',
    'tin': 'tin học',
    'sinh': 'sinh học',
    'văn': 'ngữ văn',
    'năng kiếu báo chí': 'năng khiếu báo chí',  # sửa lỗi gõ sai
}

_NOT_LETTER_AFTER = r'(?![^\W\d_])'  # chỉ đổi dấu thanh khi đứng cuối từ


def _fix_tone(s: str) -> str:
    """Đưa dấu thanh về kiểu mới: hoá -> hóa, khoẻ -> khỏe, thuý -> thúy."""
    tone_o = {'à': 'ò', 'á': 'ó', 'ả': 'ỏ', 'ã': 'õ', 'ạ': 'ọ'}
    tone_e = {'è': 'ò', 'é': 'ó', 'ẻ': 'ỏ', 'ẽ': 'õ', 'ẹ': 'ọ'}
    tone_y = {'ỳ': 'ù', 'ý': 'ú', 'ỷ': 'ủ', 'ỹ': 'ũ', 'ỵ': 'ụ'}
    s = re.sub(r'o([àáảãạ])' + _NOT_LETTER_AFTER, lambda m: tone_o[m.group(1)] + 'a', s)
    s = re.sub(r'o([èéẻẽẹ])' + _NOT_LETTER_AFTER, lambda m: tone_e[m.group(1)] + 'e', s)
    s = re.sub(r'(?<!q)u([ỳýỷỹỵ])' + _NOT_LETTER_AFTER, lambda m: tone_y[m.group(1)] + 'y', s)
    return s


def _normalize_part(part: str) -> str:
    part = part.strip()
    if not part:
        return part
    part = _fix_tone(part)
    part = SUBJECT_MAP.get(part, part)
    # Tên ngôn ngữ viết hoa: tiếng anh -> Tiếng Anh
    part = re.sub(r'\btiếng (\w+)',
                  lambda m: 'Tiếng ' + m.group(1).capitalize(), part)
    return part[0].upper() + part[1:]


def normalize_text(text):
    if pd.isna(text):
        return text

    # 1. Đồng nhất Unicode (tránh các giá trị nhìn giống nhau nhưng khác mã)
    text = unicodedata.normalize('NFC', str(text))

    # 2. Bỏ khoảng trắng đầu/cuối, gộp nhiều khoảng trắng thành 1
    text = re.sub(r'\s+', ' ', text.strip())

    # 3. Về chữ thường toàn bộ
    text = text.lower()

    # 4. Thay "lý" thành "lí"
    text = text.replace('lý', 'lí')

    # 5. Đồng nhất ký tự đặc biệt: "/" và dấu ngoặc
    text = re.sub(r'\s*/\s*', '/', text)     # "a/ b", "a / b" -> "a/b"
    text = re.sub(r'\s*\(\s*', ' (', text)   # "1( kể" -> "1 (kể"
    text = re.sub(r'\s*\)', ')', text)

    # 6. Chuẩn hóa từng môn (tách theo "/"): sửa dấu, ánh xạ, viết hoa
    return '/'.join(_normalize_part(p) for p in text.split('/'))


# Áp dụng cho các cột môn học
for col in SUBJECT_COLS:
    df[col] = df[col].apply(normalize_text)

pd.unique(df[SUBJECT_COLS].values.ravel())

array(['Toán', 'Ngữ văn', 'Tiếng Anh', nan, 'Vật lí', 'Hóa học',
       'Sinh học', 'Lịch sử', 'Địa lí', 'Tiếng Pháp', 'Tiếng Trung',
       'Tiếng Nhật', 'Gdkt&pl', 'Tiếng Nga', 'Tiếng Đức',
       'Vẽ hình họa mỹ thuật', 'Tiếng Hàn', 'Năng khiếu vẽ nghệ thuật 1',
       'Năng khiếu vẽ nghệ thuật 2', 'Vẽ mỹ thuật', 'Năng khiếu tdtt',
       'Năng khiếu thể dục thể thao', 'Năng khiếu',
       'Năng khiếu âm nhạc 1', 'Năng khiếu âm nhạc 2', 'Đọc diễn cảm',
       'Hát', 'Tin học', 'Vẽ trang trí màu', 'Vẽ năng khiếu',
       'Năng khiếu 1', 'Năng khiếu 2',
       'Nk mầm non 1 (kể chuyện, đọc, diễn cảm)', 'Nk mầm non 2 (hát)',
       'Xướng âm', 'Biểu diễn nghệ thuật', 'Năng khiếu skđa 1',
       'Năng khiếu skđa 2', 'Hình họa', 'Trang trí', 'Nk1',
       'Năng khiếu báo chí', 'Xây dựng kịch bản sự kiện',
       'Đọc kể diễn cảm', 'Hát - múa', 'Ký xướng âm',
       'Hát hoặc biểu diễn nhạc cụ', 'Công nghệ công nghiệp',
       'Công nghệ nông nghiệp', 'Năng khiếu âm nhạc',
       'Năng kh

# 4. Lưu dữ liệu đã làm sạch

In [ ]:
# export
CLEANED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(CLEANED_DATA_PATH, index=False)

In [ ]:
# Khảo sát dữ liệu sau khi lưu
print(f"Tổng số dòng sau chuẩn hóa: {len(df)}")
df.info()
df.head()

Tổng số dòng sau chuẩn hóa: 280
<class 'pandas.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   code      280 non-null    str  
 1   subject1  280 non-null    str  
 2   subject2  280 non-null    str  
 3   subject3  280 non-null    str  
 4   subject4  1 non-null      str  
dtypes: str(5)
memory usage: 21.9 KB


,code,subject1,subject2,subject3,subject4
0,D01,Toán,Ngữ văn,Tiếng Anh,NaN
1,A00,Toán,Vật lí,Hóa học,NaN
2,A01,Toán,Vật lí,Tiếng Anh,NaN
3,D07,Toán,Hóa học,Tiếng Anh,NaN
4,B00,Toán,Hóa học,Sinh học,NaN


# 5. Xuất Notebook sang Unit Script

In [ ]:
from helpers.export_to_script import export_to_script

export_to_script(output_path=SCRIPT_OUTPUT_PATH)

Đã xuất 3 cell từ 'score-blocks.ipynb' sang 'D:\UNIDOCS\IS405.R11-Big-Data\scripts-processing\unit-scripts\score_blocks.py'


'D:\\UNIDOCS\\IS405.R11-Big-Data\\scripts-processing\\unit-scripts\\score_blocks.py'